# ATML PA2 - Kaggle launcher (Task 5, RLVR vs RLAIF)

Thin launcher only. All experiment logic lives in the `.py` files.

Notebook settings before running: Accelerator **GPU T4 x2**, Internet **On**. No adapters need attaching: the RLVR and RLAIF policies are course checkpoints that section 4 downloads.

Saved responses and cached judge answers are reused, so every cell is safe to rerun. Run section 10 and download the zip after each long step.

### 1. GPU check

In [ ]:
!nvidia-smi

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%env TOKENIZERS_PARALLELISM=false

### 2. Clone or update the repo

In [ ]:
%cd /kaggle/working
!test -d atml_PA2 || git clone https://github.com/Zeeba1705/atml_PA2.git
%cd /kaggle/working/atml_PA2
!git fetch origin
!git checkout task5-feedback
!git pull
!git log --oneline -3

### 3. Install the pinned requirements

In [ ]:
!pip install -q -r requirements.txt

### 4. Course assets

Downloads only when validation fails.

In [ ]:
!python -m scripts.validate_assets || (python -m scripts.download_assets && python -m scripts.validate_assets)

### 5. Restore earlier Task 5 results (optional)

Attach the dataset made from an earlier `atml_pa2_task5.zip`. Its `results/` folder is copied in without overwriting newer files.

In [ ]:
%%bash
cd /kaggle/working/atml_PA2
mkdir -p outputs results

echo "attached inputs:"
find /kaggle/input -maxdepth 5 | head -40

#an upload that Kaggle left zipped is unpacked into the repo
for z in $(find /kaggle/input -name "*.zip"); do
    unzip -n -q "$z" -d . && echo "unzipped $z"
done

#an upload that Kaggle unpacked: every outputs/ or results/ folder found at any depth is copied in
for d in $(find /kaggle/input -type d \( -name outputs -o -name results \)); do
    cp -rn "$d/." "$(basename $d)/" && echo "restored $(basename $d) from $d"
done

ls results

### 6. Environment check and unit tests

In [ ]:
!python -m scripts.check_environment
!python -m pytest tests -q

### 7. Quick test

8 problems of each set through all three policies and the judge, written to `results/task5_feedback_quicktest/`. Run this before the full run.

In [ ]:
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm --max-examples 8
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer --max-examples 8

### 8. In-domain and transfer evaluation

Greedy responses of SFT, RLVR and RLAIF on the 300 GSM8K problems and on the 100 SVAMP problems, then the pairwise judge. The transfer run also writes `failure_types_sheet.csv` to label by hand.

In [ ]:
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm

In [ ]:
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer

### 9. Controlled diagnostic study and combined comparison

80 controlled pairs scored by the verifier and the judge, then the tables and figures.

In [ ]:
!python -m task5_feedback.score_perturbations --config configs/feedback.yaml

In [ ]:
!python -m task5_feedback.compare_feedback --config configs/feedback.yaml

### 10. Pack the results for download

Writes `/kaggle/working/atml_pa2_task5.zip`. Download it from the Output panel.

In [ ]:
%%bash
cd /kaggle/working/atml_PA2
rm -f /kaggle/working/atml_pa2_task5.zip
zip -qr /kaggle/working/atml_pa2_task5.zip results/task5_feedback results/task5_feedback_quicktest
ls -la /kaggle/working/atml_pa2_task5.zip